In [6]:
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    Trainer, TrainingArguments, DataCollatorForSeq2Seq,
    T5Config, T5ForConditionalGeneration, PreTrainedTokenizerFast
)
from transformers.trainer_utils import get_last_checkpoint

# === CONFIG ===
CSV_PATH = "./datasets/python_batch (3).csv"  # 👈 Change this each time
MODEL_DIR = "scratch_llm_model"
TOKENIZER_DIR = "scratch_tokenizer"
MAX_INPUT = 256
MAX_OUTPUT = 128
EPOCHS = 3
BATCH_SIZE = 4
VOCAB_SIZE = 32000

# === Load dataset ===
print(f"\n📦 Loading dataset: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)[["input_text", "output_text"]].dropna()
dataset = Dataset.from_pandas(df)

# === Load tokenizer ===
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    TOKENIZER_DIR,
    model_max_length=MAX_INPUT,
    bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>"
)

# === Load or create model ===
if os.path.exists(os.path.join(MODEL_DIR, "pytorch_model.bin")):
    print("🔁 Loading existing model...")
    model = T5ForConditionalGeneration.from_pretrained(MODEL_DIR)
else:
    print("🚀 Creating NEW model from scratch...")
    config = T5Config(
        vocab_size=VOCAB_SIZE,
        d_model=512,
        d_ff=2048,
        num_layers=6,
        num_heads=8,
        dropout_rate=0.1,
        eos_token_id=tokenizer.convert_tokens_to_ids("</s>"),
        pad_token_id=tokenizer.convert_tokens_to_ids("<pad>"),
        decoder_start_token_id=tokenizer.convert_tokens_to_ids("<pad>")
    )
    model = T5ForConditionalGeneration(config)

# === Tokenize ===
def tokenize(example):
    input_enc = tokenizer(
        example["input_text"], truncation=True, padding="max_length", max_length=MAX_INPUT
    )
    target_enc = tokenizer(
        example["output_text"], truncation=True, padding="max_length", max_length=MAX_OUTPUT
    )
    input_enc["labels"] = target_enc["input_ids"]
    return input_enc

dataset = dataset.map(tokenize, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# === Training args (no evaluation, no best model logic) ===
training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    eval_strategy="no",                  # ✅ Updated for future compatibility
    load_best_model_at_end=False,        # ✅ No eval dataset provided
    save_total_limit=3,
    logging_dir="./logs",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    save_safetensors=False,
    resume_from_checkpoint=True
)

# === Trainer ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# === Resume training if checkpoint exists ===
last_checkpoint = get_last_checkpoint(MODEL_DIR) if os.path.isdir(MODEL_DIR) else None
print("🚦 Starting fine-tuning...")
trainer.train(resume_from_checkpoint=last_checkpoint)

# === Save updated model & tokenizer ===
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f"✅ Done. Model fine-tuned on {CSV_PATH}")



📦 Loading dataset: ./datasets/python_batch (3).csv
🚀 Creating NEW model from scratch...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

🚦 Starting fine-tuning...


Step,Training Loss
10,3.318700
20,2.284400
30,1.813200
40,1.246300
50,0.836500
60,0.478100
70,0.291600
80,0.163800
90,0.104900
100,0.079700


✅ Done. Model fine-tuned on ./datasets/python_batch (3).csv


In [21]:
# model loading and prediction. based on training. 


import torch
import pandas as pd
from transformers import T5ForConditionalGeneration, PreTrainedTokenizerFast

# === CONFIG ===
MODEL_DIR = "scratch_llm_model"
TOKENIZER_DIR = "scratch_tokenizer"
CSV_PATH = "./datasets/python_batch (3).csv"  # ✅ Use same or different dataset
MAX_INPUT = 256
MAX_OUTPUT = 128

# === Load tokenizer and model ===
print("📦 Loading model and tokenizer...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)
model = T5ForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

# === Load dataset and pick one sample ===
df = pd.read_csv(CSV_PATH)[["input_text", "output_text"]].dropna()
# sample_row = df.sample(n=1, random_state=42).iloc[0]      this line is to select tatally random. row as the input_text while predicing. 
sample_row = df.iloc[4]   # for manually selecting any row.  index starts from 0, so 4 means 5th row.

input_text = sample_row.input_text.strip()
expected_output = sample_row.output_text.strip()

# === Tokenize input and generate output ===
inputs = tokenizer(
    input_text,
    return_tensors="pt",
    padding="max_length",
    max_length=MAX_INPUT,
    truncation=True,
    add_special_tokens=True  # ✅ Fixes missing characters like "G" in "Generates"
)

with torch.no_grad():
    generated_ids = model.generate(
        inputs["input_ids"],
        max_length=MAX_OUTPUT,
        num_beams=4,
        early_stopping=True
    )

predicted_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# === Print results ===
print("\n🧪 Input Text:\n", input_text)
print("\n✅ Expected Output:\n", expected_output)
print("\n🔮 Model Prediction:\n", predicted_output)


📦 Loading model and tokenizer...

🧪 Input Text:
 import sqlite3
conn = sqlite3.connect('test.db')
c = conn.cursor()
c.execute('CREATE TABLE IF NOT EXISTS users (id INTEGER, name TEXT)')
conn.commit()
conn.close()

✅ Expected Output:
 Creates a SQLite database and a table using the sqlite3 module.

🔮 Model Prediction:
 Creates a ite database and a table using the slite3 module.


In [23]:
import torch
from transformers import T5ForConditionalGeneration, PreTrainedTokenizerFast

# === CONFIG ===
MODEL_DIR = "scratch_llm_model"
TOKENIZER_DIR = "scratch_tokenizer"
MAX_INPUT = 256
MAX_OUTPUT = 128

# === Load model and tokenizer ===
print("📦 Loading model and tokenizer...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)
model = T5ForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

print("🤖 Your model is ready! Type 'exit' to quit.")
print("👉 Example input: for Python summarization")
print('    def greet(name):\n        print("Hello " + name)\n')

while True:
    user_input = input("\n📝 Enter input_text:\n")
    if user_input.lower() == "exit":
        print("👋 Goodbye!")
        break

    # Tokenize and generate
    inputs = tokenizer(user_input, return_tensors="pt", truncation=True, padding="max_length", max_length=MAX_INPUT)
    with torch.no_grad():
        output_ids = model.generate(
            inputs["input_ids"],
            max_length=MAX_OUTPUT,
            num_beams=4,
            early_stopping=True
        )
    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print("\n🔮 Model Prediction:\n", prediction)




"""

📦 Loading model and tokenizer...
🤖 Your model is ready! Type 'exit' to quit.

👉 Example input: for Python summarization
    def greet(name):
        print("Hello " + name)


📝 Enter input_text:
def greet(name):
    print("Hello " + name)

🔮 Model Prediction:
Function to greet a person by name.



Type any code snippet (or input_text your model is trained on)

Type exit to quit.

"""

📦 Loading model and tokenizer...
🤖 Your model is ready! Type 'exit' to quit.
👉 Example input: for Python summarization
    def greet(name):
        print("Hello " + name)




📝 Enter input_text:
  import sqlite3 conn = sqlite3.connect('test.db') c = conn.cursor() c.execute('CREATE TABLE IF NOT EXISTS users (id INTEGER, name TEXT)') conn.commit() conn.close()



🔮 Model Prediction:
 Creates a ite database and a table using the slite3 module.



📝 Enter input_text:
 exit


👋 Goodbye!


'\n\n📦 Loading model and tokenizer...\n🤖 Your model is ready! Type \'exit\' to quit.\n\n👉 Example input: for Python summarization\n    def greet(name):\n        print("Hello " + name)\n\n\n📝 Enter input_text:\ndef greet(name):\n    print("Hello " + name)\n\n🔮 Model Prediction:\nFunction to greet a person by name.\n\n\n\nType any code snippet (or input_text your model is trained on)\n\nType exit to quit.\n\n'